# Text as Data: dos hipótesis, un corpus
Doctorado en Ciencia Política — UNMSM · Métodos Cuantitativos (2026)

**Puzzle:** ¿Hasta qué punto los partidos convergen en sus propuestas pese a diferenciarse ideológicamente?
**Unidad de análisis:** un plan completo por partido; los fragmentos son unidades de procesamiento.
Pregunta → hipótesis → evidencia → representación → técnica. Buscamos explicar e interpretar diferencias discursivas, no predecir elecciones ni identificar causas.

Preparación: PDF → texto natural confiable. Abra este notebook desde la carpeta del paquete. En Colab, cargue los ocho PDFs a `planes_gobierno_pdfs` con el panel Archivos. Ejecute los bloques en orden. Los ocho PDFs originales se conservan sin cambios. Si no están presentes, el primer bloque los descarga desde una revisión fija del repositorio del curso.

## preTEXT-01: localizar o descargar los PDF
Instalación mínima: pypdf para extraer texto. Si utiliza enlaces, deben apuntar directamente a PDFs accesibles. Complete el manifiesto con los nombres reales; no cambie las identidades para obtener un orden esperado.

In [1]:
%pip -q install pypdf

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import csv, json, re, hashlib, unicodedata
from urllib.request import urlopen
ROOT = Path.cwd()
PDFS = ROOT / 'planes_gobierno_pdfs'
PDFS.mkdir(exist_ok=True)
# Cambie solo los nombres de archivo si los originales tienen otros nombres.
MANIFIESTO = [{'id_partido': 'A',
  'partido': 'Renovación Popular Social',
  'candidato': 'Candidata A',
  'archivo': 'Partido_A_plan_gobierno.pdf',
  'url': 'https://raw.githubusercontent.com/doctorado-cuanticp/text/cb69bdd8d7c39de9eee4677d7e2b600ba87c044a/planes_gobierno/Partido_A_plan_gobierno.pdf'},
 {'id_partido': 'B',
  'partido': 'Frente Progresista',
  'candidato': 'Candidato B',
  'archivo': 'Partido_B_plan_gobierno.pdf',
  'url': 'https://raw.githubusercontent.com/doctorado-cuanticp/text/cb69bdd8d7c39de9eee4677d7e2b600ba87c044a/planes_gobierno/Partido_B_plan_gobierno.pdf'},
 {'id_partido': 'C',
  'partido': 'Movimiento Democrático',
  'candidato': 'Candidata C',
  'archivo': 'Partido_C_plan_gobierno.pdf',
  'url': 'https://raw.githubusercontent.com/doctorado-cuanticp/text/cb69bdd8d7c39de9eee4677d7e2b600ba87c044a/planes_gobierno/Partido_C_plan_gobierno.pdf'},
 {'id_partido': 'D',
  'partido': 'Centro Cívico',
  'candidato': 'Candidato D',
  'archivo': 'Partido_D_plan_gobierno.pdf',
  'url': 'https://raw.githubusercontent.com/doctorado-cuanticp/text/cb69bdd8d7c39de9eee4677d7e2b600ba87c044a/planes_gobierno/Partido_D_plan_gobierno.pdf'},
 {'id_partido': 'E',
  'partido': 'Alianza Nacional',
  'candidato': 'Candidata E',
  'archivo': 'Partido_E_plan_gobierno.pdf',
  'url': 'https://raw.githubusercontent.com/doctorado-cuanticp/text/cb69bdd8d7c39de9eee4677d7e2b600ba87c044a/planes_gobierno/Partido_E_plan_gobierno.pdf'},
 {'id_partido': 'F',
  'partido': 'Libertad Republicana',
  'candidato': 'Candidato F',
  'archivo': 'Partido_F_plan_gobierno.pdf',
  'url': 'https://raw.githubusercontent.com/doctorado-cuanticp/text/cb69bdd8d7c39de9eee4677d7e2b600ba87c044a/planes_gobierno/Partido_F_plan_gobierno.pdf'},
 {'id_partido': 'G',
  'partido': 'Futuro Liberal',
  'candidato': 'Candidata G',
  'archivo': 'Partido_G_plan_gobierno.pdf',
  'url': 'https://raw.githubusercontent.com/doctorado-cuanticp/text/cb69bdd8d7c39de9eee4677d7e2b600ba87c044a/planes_gobierno/Partido_G_plan_gobierno.pdf'},
 {'id_partido': 'H',
  'partido': 'Mercado y Libertad',
  'candidato': 'Candidato H',
  'archivo': 'Partido_H_plan_gobierno.pdf',
  'url': 'https://raw.githubusercontent.com/doctorado-cuanticp/text/cb69bdd8d7c39de9eee4677d7e2b600ba87c044a/planes_gobierno/Partido_H_plan_gobierno.pdf'}]
for r in MANIFIESTO:
    p = PDFS / r['archivo']
    if not p.exists() and r['url']:
        with urlopen(r['url'], timeout=60) as response:
            data = response.read()
        if not data.startswith(b'%PDF-'):
            raise ValueError('El enlace no devolvió un PDF: ' + r['url'])
        p.write_bytes(data)
missing = [r['archivo'] for r in MANIFIESTO if not (PDFS/r['archivo']).exists()]
assert not missing, f'Faltan PDFs originales: {missing}. Cárguelos o ajuste el manifiesto.'
assert len({r['id_partido'] for r in MANIFIESTO}) == 8

## preTEXT-02: extracción y diagnóstico por página
**Cuidado:** texto vacío puede indicar un escaneo. Este flujo no hace OCR. Revise también columnas, tablas y orden de lectura contra el PDF.

In [3]:
from pypdf import PdfReader
paginas = []
for r in MANIFIESTO:
    reader = PdfReader(PDFS/r['archivo'])
    for k, page in enumerate(reader.pages, 1):
        text = page.extract_text() or ''
        paginas.append(dict(id_partido=r['id_partido'], archivo=r['archivo'], pagina=k, original=text))
        print(r['id_partido'], k, len(text), 'caracteres', repr(text[:100]))
assert all(p['original'].strip() for p in paginas), 'Hay páginas vacías: revisar OCR/extracción antes de continuar.'

A 1 139 caracteres 'PLAN DE GOBIERNO\n Renovación Popular Social\n Candidatura: Candidata A\n Documento ficticio elaborado '
A 2 789 caracteres 'Diagnóstico y prioridades\nNuestro plan propone crecimiento económico con empleo digno, seguridad ciu'
B 1 132 caracteres 'PLAN DE GOBIERNO\n Frente Progresista\n Candidatura: Candidato B\n Documento ficticio elaborado exclusi'
B 2 731 caracteres 'Diagnóstico y prioridades\nNuestro plan propone crecimiento económico con empleo digno, seguridad ciu'
C 1 136 caracteres 'PLAN DE GOBIERNO\n Movimiento Democrático\n Candidatura: Candidata C\n Documento ficticio elaborado exc'
C 2 744 caracteres 'Diagnóstico y prioridades\nNuestro plan propone crecimiento económico sostenible, empleo formal, segu'
D 1 127 caracteres 'PLAN DE GOBIERNO\n Centro Cívico\n Candidatura: Candidato D\n Documento ficticio elaborado exclusivamen'
D 2 749 caracteres 'Diagnóstico y prioridades\nNuestro plan propone crecimiento económico sostenible, empleo formal, segu'
E 1 130 

## preTEXT-03: limpiar artefactos, conservar lenguaje
**Clave:** conservar mayúsculas, negaciones, puntuación y palabras funcionales. La limpieza no es BoW. Declare encabezados/pies exactos solo después de revisarlos; no elimine automáticamente líneas repetidas que podrían ser propuestas. Los guiones de final de línea requieren inspección.

In [4]:
# Por ejemplo: {'A': ['encabezado exacto verificado']}. No añadir propuestas.
LINEAS_EXCLUIDAS = {'A': ['PLAN DE GOBIERNO',
       'Renovación Popular Social',
       'Candidatura: Candidata A',
       'Documento ficticio elaborado exclusivamente para fines pedagógicos.',
       'Diagnóstico y prioridades',
       'Economía y empleo',
       'Políticas sociales',
       'Instituciones y territorio'],
 'B': ['PLAN DE GOBIERNO',
       'Frente Progresista',
       'Candidatura: Candidato B',
       'Documento ficticio elaborado exclusivamente para fines pedagógicos.',
       'Diagnóstico y prioridades',
       'Economía y empleo',
       'Políticas sociales',
       'Instituciones y territorio'],
 'C': ['PLAN DE GOBIERNO',
       'Movimiento Democrático',
       'Candidatura: Candidata C',
       'Documento ficticio elaborado exclusivamente para fines pedagógicos.',
       'Diagnóstico y prioridades',
       'Economía y empleo',
       'Políticas sociales',
       'Instituciones y territorio'],
 'D': ['PLAN DE GOBIERNO',
       'Centro Cívico',
       'Candidatura: Candidato D',
       'Documento ficticio elaborado exclusivamente para fines pedagógicos.',
       'Diagnóstico y prioridades',
       'Economía y empleo',
       'Políticas sociales',
       'Instituciones y territorio'],
 'E': ['PLAN DE GOBIERNO',
       'Alianza Nacional',
       'Candidatura: Candidata E',
       'Documento ficticio elaborado exclusivamente para fines pedagógicos.',
       'Diagnóstico y prioridades',
       'Economía y empleo',
       'Políticas sociales',
       'Instituciones y territorio'],
 'F': ['PLAN DE GOBIERNO',
       'Libertad Republicana',
       'Candidatura: Candidato F',
       'Documento ficticio elaborado exclusivamente para fines pedagógicos.',
       'Diagnóstico y prioridades',
       'Economía y empleo',
       'Políticas sociales',
       'Instituciones y territorio'],
 'G': ['PLAN DE GOBIERNO',
       'Futuro Liberal',
       'Candidatura: Candidata G',
       'Documento ficticio elaborado exclusivamente para fines pedagógicos.',
       'Diagnóstico y prioridades',
       'Economía y empleo',
       'Políticas sociales',
       'Instituciones y territorio'],
 'H': ['PLAN DE GOBIERNO',
       'Mercado y Libertad',
       'Candidatura: Candidato H',
       'Documento ficticio elaborado exclusivamente para fines pedagógicos.',
       'Diagnóstico y prioridades',
       'Economía y empleo',
       'Políticas sociales',
       'Instituciones y territorio']}
# Correcciones literales verificadas contra los PDFs, por partido.
REEMPLAZOS = {}
def limpiar(text, partido):
    text = unicodedata.normalize('NFC', text).replace('\u00ad', '')
    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if line not in LINEAS_EXCLUIDAS.get(partido, [])]
    text = '\n'.join(lines)
    for old, new in REEMPLAZOS.get(partido, {}).items():
        text = text.replace(old, new)
    return re.sub(r'\s+', ' ', text).strip()
for p in paginas:
    p['texto'] = limpiar(p['original'], p['id_partido'])
    p['posible_guion_cortado'] = bool(re.search(r'\w-\s*\n\s*\w', p['original']))
    p['caracteres_reemplazo'] = p['texto'].count('�')
    print(p['id_partido'], p['pagina'], len(p['texto'].split()), p['posible_guion_cortado'], p['caracteres_reemplazo'])

A 1 0 False 0
A 2 84 False 0
B 1 0 False 0
B 2 81 False 0
C 1 0 False 0
C 2 80 False 0
D 1 0 False 0
D 2 78 False 0
E 1 0 False 0
E 2 79 False 0
F 1 0 False 0
F 2 83 False 0
G 1 0 False 0
G 2 87 False 0
H 1 0 False 0
H 2 85 False 0


## preTEXT-04: comparación y aprobación de calidad
Compare cada página con su PDF, especialmente negaciones, títulos, tablas y uniones entre líneas. Registre decisiones y cambie la bandera solo al terminar. Esta revisión humana evita tratar extracción defectuosa como evidencia.

In [5]:
for p in paginas:
    print('\n', p['id_partido'], 'página', p['pagina'])
    print('ORIGINAL:', p['original'][:800])
    print('PREPARADO:', p['texto'][:800])
REVISION_COMPLETA = True  # Revisión del corpus original incluido; repetir si cambia el PDF.
NOTA_REVISION = 'Ocho PDFs digitales de dos páginas: portada en página 1; cinco oraciones programáticas en página 2. Se excluyen portada y cuatro títulos exactos; se conservan todas las propuestas. Sin OCR ni correcciones léxicas. Revisado contra PDF y extracción.'  # Qué se revisó, qué se corrigió y limitaciones restantes.


 A página 1
ORIGINAL: PLAN DE GOBIERNO
 Renovación Popular Social
 Candidatura: Candidata A
 Documento ficticio elaborado exclusivamente para fines pedagógicos.

PREPARADO: 

 A página 2
ORIGINAL: Diagnóstico y prioridades
Nuestro plan propone crecimiento económico con empleo digno, seguridad ciudadana,
educación pública de calidad, salud universal y lucha contra la corrupción.
Impulsaremos vivienda social, descentralización, protección ambiental y servicios públicos
accesibles en todo el territorio.
Economía y empleo
El desarrollo nacional requiere un Estado activo que garantice derechos sociales, fortalezca
empresas públicas estratégicas y reduzca la desigualdad.
Políticas sociales
Promoveremos inversión pública, reforma tributaria progresiva, negociación colectiva,
protección laboral y mayor participación de trabajadores.
Instituciones y territorio
La inversión privada podrá contribuir al desarrollo cuando respete regulación, derechos
laborales y objetivos nacionales.

PREPARADO: N

## preTEXT-05: guardar corpus y trazabilidad
Se guardan texto por página, texto completo, huellas digitales de los PDFs y decisiones de limpieza. H2 exige leer A y H y justificar su interpretación; sus etiquetas no proceden del modelo.

In [6]:
assert REVISION_COMPLETA and NOTA_REVISION.strip(), 'Complete preTEXT-04 antes de exportar.'
assert all(p['texto'].strip() or p['pagina']==1 for p in paginas), 'Página de contenido vacía.'
records = []
for r in MANIFIESTO:
    text = ' '.join(p['texto'] for p in paginas if p['id_partido'] == r['id_partido'] and p['texto'])
    records.append({'id_partido': r['id_partido'], 'partido':r['partido'], 'candidato':r['candidato'], 'archivo': r['archivo'], 'url':r['url'], 'texto': text,
                    'sha256_pdf': hashlib.sha256((PDFS/r['archivo']).read_bytes()).hexdigest()})
assert len({r['texto'] for r in records}) == 8, 'Hay planes duplicados.'
with open(ROOT/'planes_gobierno_preparados.csv', 'w', encoding='utf-8', newline='') as f:
    w=csv.DictWriter(f, fieldnames=records[0].keys()); w.writeheader(); w.writerows(records)
(ROOT/'paginas_preparadas.json').write_text(json.dumps(paginas, ensure_ascii=False, indent=2), encoding='utf-8')
(ROOT/'revision_corpus.json').write_text(json.dumps({'nota': NOTA_REVISION, 'exclusiones': LINEAS_EXCLUIDAS, 'reemplazos': REEMPLAZOS}, ensure_ascii=False, indent=2), encoding='utf-8')
print('Corpus preparado: 8 planes. Continúe con TextAnalysis.ipynb.')

Corpus preparado: 8 planes. Continúe con TextAnalysis.ipynb.
